In [0]:
import pandas as pd                                                   # Import pandas for data cleaning
import numpy as np                                                    # Import Numpy for Maths functions
import matplotlib.pyplot as plt                                       # Import Matplot for Viz functions
import seaborn as sns                                                 # Import Seaborn for visualization
import matplotlib.pyplot as plt                                       # Import matplotlib library for visualization
import plotly.express as px                                           # Import plotly library for visualization
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest                          # EDA-Isolation Forest Analysis Functions

from pyspark.sql import functions as F
from pyspark.sql.functions import col, StringType, NumericType                  # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev, count, sum as _sum    # MathsFunctions
from pyspark.sql.functions import to_date, year, month, datediff                # DateFunctions
from pyspark.sql.functions import abs                                           # OtherFunctions

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

from sklearn.linear_model import LinearRegression                              # LinearRegression Analysis Functions
from sklearn.metrics import r2_score, mean_squared_error                       # LinearRegression Analysis Functions

from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder    # Classification Analysis Functions
from pyspark.ml import Pipeline                                                 # Classification Analysis Functions
from sklearn.linear_model import LogisticRegression                             # Classification Analysis Functions
from sklearn.metrics import mean_squared_error, r2_score                        # Classification Analysis Functions
from sklearn.impute import SimpleImputer                                        # Classification Analysis Functions
from sklearn.preprocessing import LabelEncoder                                  # Classification Analysis Functions
from sklearn.preprocessing import OneHotEncoder                                 # Classification Analysis Functions
from sklearn.preprocessing import StandardScaler                                # Classification Analysis Functions
from sklearn.model_selection import train_test_split                            # Classification Analysis Functions
from sklearn.metrics import accuracy_score, confusion_matrix                    # Classification Analysis Functions
from sklearn.metrics import classification_report                               # Classification Analysis Functions
from sklearn.metrics import roc_curve, auc                                      # Classification Analysis Functions

from sklearn.tree import DecisionTreeClassifier                                 # DecisionTree Analysis Functions

from sklearn.ensemble import RandomForestClassifier                             # RandomForest Analysis Functions

from sklearn.cluster import KMeans                                              # KMeans Cluster Analysis Functions


In [0]:
from pyspark.sql.functions import col

# List all tables in contoso.bronze schema
tables_df = spark.sql("SHOW TABLES IN contoso.bronze").filter(col("isTemporary") == False)

# Collect table names using DataFrame .collect() directly
tables = [row.tableName for row in tables_df.collect()]

# Read each table and print schema
for table in tables:
    df = spark.read.table(f"contoso.bronze.{table}")
    print(f"Schema for table: contoso.bronze.{table}")
    df.printSchema()


In [0]:
# FORMATING THE COLUMNS NAMES  
    # For Order table : replace spaces with underscores, lowercase
clean_cols = [c.replace(" ", "_").lower() for c in orders.columns]
for old, new in zip(orders.columns, clean_cols):
    orders = orders.withColumnRenamed(old, new)

    # For people table : replace spaces with underscores, lowercase
clean_cols = [c.replace(" ", "_").lower() for c in people.columns]
for old, new in zip(people.columns, clean_cols):
    people = people.withColumnRenamed(old, new)

    # For returns table : replace spaces with underscores, lowercase
clean_cols = [c.replace(" ", "_").lower() for c in returns.columns]
for old, new in zip(returns.columns, clean_cols):
    returns = returns.withColumnRenamed(old, new)

# display(summary_df)

In [0]:
# Cell 3: Summary statistics for Orders table
from pyspark.sql.functions import col, mean, min, max, stddev, count, when, expr

# Example: Sales, Profit, Quantity columns
summary_orders = orders.select(
    mean(col("Sales")).alias("Sales_Mean"),
    expr("percentile_approx(Sales, 0.5)").alias("Sales_Median"),
    min(col("Sales")).alias("Sales_Min"),
    max(col("Sales")).alias("Sales_Max"),
    stddev(col("Sales")).alias("Sales_StdDev"),
    count(when(col("Sales").isNull(), 1)).alias("Sales_Nulls"),

    mean(col("Profit")).alias("Profit_Mean"),
    expr("percentile_approx(Profit, 0.5)").alias("Profit_Median"),
    min(col("Profit")).alias("Profit_Min"),
    max(col("Profit")).alias("Profit_Max"),
    stddev(col("Profit")).alias("Profit_StdDev"),
    count(when(col("Profit").isNull(), 1)).alias("Profit_Nulls"),

    mean(col("Quantity")).alias("Quantity_Mean"),
    expr("percentile_approx(Quantity, 0.5)").alias("Quantity_Median"),
    min(col("Quantity")).alias("Quantity_Min"),
    max(col("Quantity")).alias("Quantity_Max"),
    stddev(col("Quantity")).alias("Quantity_StdDev"),
    count(when(col("Quantity").isNull(), 1)).alias("Quantity_Nulls")
)
display(summary_orders)

In [0]:
# Build summary for each column
columns = orders.columns
summary_data = []

# Get total record count once
total_records = orders.count()

for column in columns:
    not_nulls = orders.filter(F.col(column).isNotNull()).count()
    nulls = orders.filter(F.col(column).isNull()).count()
    summary_data.append((column, total_records, not_nulls, nulls))

summary_df = spark.createDataFrame(
    summary_data,
    ["Column", "AllValues", "NotNulls", "Nulls"]
)

# Filter columns with nulls > 0
summary_df = summary_df.filter(col("Nulls") > 0)

# Display summary ordered by Nulls descending
display(summary_df.orderBy("Nulls", ascending=False))

# Replace nulls with 0 in Spark DataFrame
# df = orders.na.fill(0)

# Print total number of columns with nulls
print("Columns with Nulls:", summary_df.count())

In [0]:

# Create temp table for Outlier Stats
Outliers_Orders = orders.select("Order ID", "Sales")

# Calculate quartiles and fences using approxQuantile
Q1 = Outliers_Orders.approxQuantile("Sales", [0.25], 0.01)[0]
Q3 = Outliers_Orders.approxQuantile("Sales", [0.75], 0.01)[0]
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Add SalesOutlier column (1 = Yes, 0 = No) → keep all rows
Outliers_Orders = Outliers_Orders.withColumn(
    "SalesOutlier",
    F.when((F.col("Sales") < lower_fence) | (F.col("Sales") > upper_fence), F.lit(1)).otherwise(F.lit(0))
)

# Compute mean and stddev
stats = Outliers_Orders.select(
    F.mean("Sales").alias("mean"),
    F.stddev("Sales").alias("stddev")
).collect()[0]

mean_sales = stats["mean"]
std_sales = stats["stddev"]

# Add Z-score and anomaly flag
Outliers_Orders = Outliers_Orders.withColumn(
    "Sales_Zscore", (F.col("Sales") - mean_sales) / std_sales
).withColumn(
    "SalesAnomaly", F.when(F.abs(F.col("Sales_Zscore")) > 3, 1).otherwise(0)
)

display(Outliers_Orders)


In [0]:
def fill_nulls_with_mode_mean(df):
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            mode_row = df.groupBy(col_name).count().orderBy(F.desc("count")).first()
            if mode_row is not None:
                mode_val = mode_row[0]
                df = df.fillna({col_name: mode_val})
        elif dtype in ["double", "int", "long", "float", "decimal"]:
            mean_val = df.select(F.mean(F.col(col_name))).first()[0]
            if mean_val is not None:
                df = df.fillna({col_name: mean_val})
    return df

df_orders  = fill_nulls_with_mode_mean(orders)
df_people  = fill_nulls_with_mode_mean(people)
df_returns = fill_nulls_with_mode_mean(returns)


# Force consistent casing for all column names
df_returns = df_returns.toDF(*[c.lower() for c in df_returns.columns])

# If both exist, drop one
if "Returned" in df_returns.columns and "returned" in df_returns.columns:
    df_returns = df_returns.drop("returned")

df_orders.write.mode("overwrite").option("schemaOverwrite", "true").saveAsTable("samplesuperstore.silverdata.orders")
df_people.write.mode("overwrite").option("schemaOverwrite", "true").saveAsTable("samplesuperstore.silverdata.people")
df_returns.write.mode("overwrite").option("schemaOverwrite", "true").saveAsTable("samplesuperstore.silverdata.returns")